# 02 — Rendering Checks

Validate the PDF → PNG rendering pipeline before any training run.

This notebook verifies:
- DPI impact on visual quality and image size
- Grayscale vs RGB rendering
- Aspect-ratio preservation and white padding to 224×224
- Edge-case files from both source directories
- Batch render timing and `skip_existing` cache logic

**Prerequisites**: run `01_data_audit.ipynb` at least once so you know which PDFs are available.

In [ ]:
# Setup: imports and project root
import sys
import time
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add project root to sys.path so src/ imports work
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
RENDERED_DIR = DATA_DIR / 'rendered_pages'
RENDERED_DIR.mkdir(parents=True, exist_ok=True)

# Import rendering utilities from src/
from src.data.render_pdf import render_page, render_all

print(f'Project root   : {PROJECT_ROOT}')
print(f'render_page    : {render_page}')
print(f'render_all     : {render_all}')

In [ ]:
# Discover all PDF files to have a pool to sample from
EXCLUDE_DIRS = {'rendered_pages', 'splits'}
subdirs = [d for d in DATA_DIR.iterdir() if d.is_dir() and d.name not in EXCLUDE_DIRS]

all_pdfs = []
per_subdir: dict[str, list[Path]] = {}
for subdir in subdirs:
    pdfs = sorted(subdir.rglob('*.pdf'))
    per_subdir[subdir.name] = pdfs
    all_pdfs.extend(pdfs)

print(f'Total PDFs available: {len(all_pdfs)}')
for name, pdfs in per_subdir.items():
    print(f'  {name}: {len(pdfs)} files')

if not all_pdfs:
    print('\nWARNING: No PDF files found. Check that data subdirectories are populated.')
    print('The rest of this notebook requires at least a few PDFs to be present.')

## 1. DPI Comparison

Render 3 randomly sampled PDFs at DPI 100, 150, and 200. Display side-by-side to confirm that:
- Higher DPI yields sharper text
- All outputs are padded to exactly 224×224 regardless of DPI
- The rendering is fast enough at 150 DPI (the training default)

In [ ]:
# Render 3 sample PDFs at 3 DPI levels and display them in a grid
random.seed(42)
DPIS = [100, 150, 200]
N_SAMPLES = min(3, len(all_pdfs))

if N_SAMPLES == 0:
    print('No PDFs available — skipping DPI comparison.')
else:
    sample_pdfs = random.sample(all_pdfs, N_SAMPLES)

    fig, axes = plt.subplots(N_SAMPLES, len(DPIS), figsize=(4 * len(DPIS), 4 * N_SAMPLES))
    # Ensure axes is always 2-D
    if N_SAMPLES == 1:
        axes = [axes]

    for row_idx, pdf_path in enumerate(sample_pdfs):
        for col_idx, dpi in enumerate(DPIS):
            ax = axes[row_idx][col_idx]
            try:
                img = render_page(str(pdf_path), dpi=dpi, grayscale=True, target_size=(224, 224))
                ax.imshow(img, cmap='gray', vmin=0, vmax=255)
                ax.set_title(f'DPI={dpi}  size={img.size}', fontsize=9)
            except Exception as e:
                ax.text(0.5, 0.5, f'Error:\n{e}', ha='center', va='center',
                        transform=ax.transAxes, fontsize=8, color='red')
            ax.axis('off')
            if col_idx == 0:
                # Show truncated filename on the leftmost column
                short_name = pdf_path.name[:40] + ('…' if len(pdf_path.name) > 40 else '')
                ax.set_ylabel(short_name, fontsize=7, labelpad=4)
                ax.yaxis.set_visible(True)
                ax.tick_params(left=False, labelleft=False)

    fig.suptitle('DPI Comparison: 100 vs 150 vs 200  (all padded to 224×224)', fontsize=12)
    plt.tight_layout()
    plt.show()

## 2. Grayscale vs RGB

Render the same PDF in both modes and compare the outputs.

In [ ]:
# Compare grayscale and RGB rendering for a single PDF
if not all_pdfs:
    print('No PDFs available — skipping grayscale vs RGB comparison.')
else:
    test_pdf = all_pdfs[0]
    print(f'Test file: {test_pdf.name}')

    img_gray = render_page(str(test_pdf), dpi=150, grayscale=True, target_size=(224, 224))
    img_rgb  = render_page(str(test_pdf), dpi=150, grayscale=False, target_size=(224, 224))

    print(f'Grayscale mode: {img_gray.mode}, size: {img_gray.size}')
    print(f'RGB mode      : {img_rgb.mode},  size: {img_rgb.size}')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
    ax1.imshow(img_gray, cmap='gray', vmin=0, vmax=255)
    ax1.set_title(f'Grayscale (mode={img_gray.mode})')
    ax1.axis('off')

    ax2.imshow(img_rgb)
    ax2.set_title(f'RGB (mode={img_rgb.mode})')
    ax2.axis('off')

    fig.suptitle('Grayscale vs RGB — same PDF, DPI=150', fontsize=11)
    plt.tight_layout()
    plt.show()

## 3. Aspect-Ratio Preservation Check

Verify that all rendered images are exactly 224×224 and that non-square PDFs are padded (not stretched).

In [ ]:
# Check aspect-ratio preservation: output must always be exactly (224, 224)
import numpy as np

TARGET_SIZE = (224, 224)
N_ASPECT_CHECK = min(10, len(all_pdfs))

if N_ASPECT_CHECK == 0:
    print('No PDFs available — skipping aspect-ratio check.')
else:
    sample_aspect = all_pdfs[:N_ASPECT_CHECK]
    results = []
    for pdf_path in sample_aspect:
        try:
            img = render_page(str(pdf_path), dpi=150, grayscale=True, target_size=TARGET_SIZE)
            size_ok = img.size == TARGET_SIZE

            # Check for white padding: at least one border row/column should be nearly white
            arr = np.array(img)
            top_row_mean   = arr[0, :].mean()
            bot_row_mean   = arr[-1, :].mean()
            left_col_mean  = arr[:, 0].mean()
            right_col_mean = arr[:, -1].mean()
            border_means = [top_row_mean, bot_row_mean, left_col_mean, right_col_mean]
            has_padding = any(m > 240 for m in border_means)  # white = 255

            results.append({
                'file': pdf_path.name[:50],
                'size': img.size,
                'size_ok': size_ok,
                'has_white_border': has_padding,
                'border_max_mean': round(max(border_means), 1),
            })
        except Exception as e:
            results.append({'file': pdf_path.name[:50], 'size': None,
                            'size_ok': False, 'has_white_border': False,
                            'border_max_mean': None})
            print(f'Error rendering {pdf_path.name}: {e}')

    import pandas as pd
    res_df = pd.DataFrame(results)
    all_ok = res_df['size_ok'].all()
    print(f'All outputs exactly {TARGET_SIZE}: {all_ok} ✓' if all_ok else f'FAIL: some outputs are wrong size!')
    print()
    display(res_df)

## 4. Edge-Case Grid

Render representative PDFs from **both** source directories in a 4×4 grid to visually inspect the variety of document types.

In [ ]:
# 4x4 grid: 2 PDFs per row from each source directory (up to 8 per dir)
GRID_ROWS, GRID_COLS = 4, 4
N_GRID = GRID_ROWS * GRID_COLS  # 16 total

# Sample evenly from available subdirectories
grid_pdfs = []
subdir_keys = list(per_subdir.keys())
if subdir_keys:
    random.seed(7)
    per_dir_count = N_GRID // max(len(subdir_keys), 1)
    for k in subdir_keys:
        pool = per_subdir[k]
        grid_pdfs.extend(random.sample(pool, min(per_dir_count, len(pool))))
    # Pad with extra samples if fewer than N_GRID
    while len(grid_pdfs) < N_GRID and all_pdfs:
        grid_pdfs.append(random.choice(all_pdfs))
    grid_pdfs = grid_pdfs[:N_GRID]

if not grid_pdfs:
    print('No PDFs available — skipping edge-case grid.')
else:
    fig, axes = plt.subplots(GRID_ROWS, GRID_COLS,
                             figsize=(GRID_COLS * 2.5, GRID_ROWS * 2.5))
    axes_flat = axes.flatten()

    for ax, pdf_path in zip(axes_flat, grid_pdfs):
        try:
            img = render_page(str(pdf_path), dpi=100, grayscale=True, target_size=(224, 224))
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
            short = pdf_path.name[:22] + '…' if len(pdf_path.name) > 22 else pdf_path.name
            ax.set_title(short, fontsize=6)
        except Exception as e:
            ax.text(0.5, 0.5, f'Error:\n{str(e)[:40]}', ha='center', va='center',
                    transform=ax.transAxes, fontsize=6, color='red')
        ax.axis('off')

    # Hide any unused axes
    for ax in axes_flat[len(grid_pdfs):]:
        ax.set_visible(False)

    fig.suptitle('Edge-Case Grid: sample pages from both source directories', fontsize=11)
    plt.tight_layout()
    plt.show()

## 5. Batch Render Timing

Time `render_all` on a small subset (up to 10 PDFs) and extrapolate the expected total render time for the full dataset.

In [ ]:
# Batch render timing: render up to 10 PDFs and estimate full-dataset time
import tempfile, shutil

N_TIMING = min(10, len(all_pdfs))

if N_TIMING == 0:
    print('No PDFs available — skipping batch render timing.')
else:
    # Use a temporary directory so we don't permanently render during the notebook run
    tmp_dir = Path(tempfile.mkdtemp(prefix='render_timing_'))
    timing_pdfs = all_pdfs[:N_TIMING]

    # Copy subset to a temp source directory to use render_all cleanly
    tmp_src = tmp_dir / 'src'
    tmp_out = tmp_dir / 'out'
    tmp_src.mkdir()
    for p in timing_pdfs:
        shutil.copy2(p, tmp_src / p.name)

    t_start = time.perf_counter()
    results = render_all(
        pdf_dir=str(tmp_src),
        output_dir=str(tmp_out),
        dpi=150,
        grayscale=True,
        target_size=(224, 224),
        skip_existing=False,
    )
    elapsed = time.perf_counter() - t_start

    n_rendered = sum(1 for r in results if r['status'] == 'rendered')
    n_errors   = sum(1 for r in results if r['status'] == 'error')
    per_pdf_sec = elapsed / max(n_rendered, 1)

    total_pdfs = len(all_pdfs)
    estimated_total_min = (per_pdf_sec * total_pdfs) / 60

    print(f'Rendered {n_rendered}/{N_TIMING} PDFs in {elapsed:.2f}s  ({n_errors} errors)')
    print(f'Average time per PDF  : {per_pdf_sec:.3f}s')
    print(f'Total PDFs in dataset : {total_pdfs}')
    print(f'Estimated total time  : {estimated_total_min:.1f} minutes  ({estimated_total_min/60:.2f} hours)')

    # Clean up temp directory
    shutil.rmtree(tmp_dir, ignore_errors=True)

## 6. Cache / Skip-Existing Check

Verify that `render_all` with `skip_existing=True` correctly skips already-rendered files on a second run, making the pipeline idempotent.

In [ ]:
# Skip-existing (cache) check: render the same files twice
import tempfile, shutil

N_CACHE_CHECK = min(5, len(all_pdfs))

if N_CACHE_CHECK == 0:
    print('No PDFs available — skipping cache check.')
else:
    tmp_dir = Path(tempfile.mkdtemp(prefix='render_cache_'))
    tmp_src = tmp_dir / 'src'
    tmp_out = tmp_dir / 'out'
    tmp_src.mkdir()
    for p in all_pdfs[:N_CACHE_CHECK]:
        shutil.copy2(p, tmp_src / p.name)

    # First pass: render everything
    t0 = time.perf_counter()
    r1 = render_all(str(tmp_src), str(tmp_out), skip_existing=False)
    t1 = time.perf_counter() - t0

    # Second pass: skip_existing=True — should be near-instant
    t0 = time.perf_counter()
    r2 = render_all(str(tmp_src), str(tmp_out), skip_existing=True)
    t2 = time.perf_counter() - t0

    first_statuses  = [r['status'] for r in r1]
    second_statuses = [r['status'] for r in r2]

    print('=== First pass (skip_existing=False) ===')
    print(f'  rendered : {first_statuses.count("rendered")}')
    print(f'  skipped  : {first_statuses.count("skipped")}')
    print(f'  errors   : {first_statuses.count("error")}')
    print(f'  time     : {t1:.3f}s')

    print('\n=== Second pass (skip_existing=True) ===')
    print(f'  rendered : {second_statuses.count("rendered")}')
    print(f'  skipped  : {second_statuses.count("skipped")}')
    print(f'  errors   : {second_statuses.count("error")}')
    print(f'  time     : {t2:.3f}s')

    if second_statuses.count('skipped') == first_statuses.count('rendered'):
        print('\nskip_existing=True correctly skipped all previously rendered files. ✓')
    else:
        print('\nWARNING: skip_existing behaviour unexpected — check output directory mirroring.')

    shutil.rmtree(tmp_dir, ignore_errors=True)

## 7. Rendering Recommendations

### Settings to use for training
| Parameter | Value | Rationale |
|---|---|---|
| `dpi` | **150** | Good text legibility; ~2× faster than DPI 300; matches DiT pretraining resolution |
| `grayscale` | **True** | Document images carry almost no colour signal; saves memory and avoids colour-normalisation issues |
| `target_size` | **(224, 224)** | Standard input size for ViT-Base, ResNet50, and DiT |
| `skip_existing` | **True** | Renders are deterministic — always safe to skip and re-run idempotently |

### Known quirks
- Hebrew filenames must be handled as UTF-8 `pathlib.Path` objects — avoid `str` concatenation
- PDF pages may have very wide or very tall aspect ratios; white padding is centred (confirmed above)
- Scanned PDFs at low scan resolution may appear blurry even at DPI 200 — this is inherent to the source

### Next step
Run `render_all` on the **full** `data/` tree once PDFs are in place, then proceed to annotation and `03_label_consistency.ipynb`.